# 机器学习与因子投资完整教程：从线性模型到树模型

## 📚 教学目标
1. 理解机器学习在因子投资中的三个核心应用：**预测、特征选择、非线性建模**
2. 掌握 **正则化方法**（Ridge / LASSO / ElasticNet）的原理与因子选择功能
3. 使用 **树模型**（随机森林 / GBDT）捕捉因子的非线性效应和交互作用
4. 理解金融领域 **交叉验证** 的特殊性（Purged K-Fold）和样本外评估指标
5. 对比线性模型 vs 树模型的预测能力（IC、ICIR、$R^2_{oos}$）

**参考书**：《因子投资：方法与实践》（石川）第 6.8 节
**教学策略**：先用 10 只股票的微型数据集手算每一步，再扩展到 300 只 × 60 月的完整模拟

---

## 1. 为什么因子投资需要机器学习？

### 🎯 场景设定

经典因子投资的核心任务是：**用公司特征预测股票未来收益率的截面差异**。

传统方法（排序法、Fama-MacBeth 回归）存在两个瓶颈：

```
瓶颈 1: 线性假设 —— 真实的特征-收益关系可能是非线性的
瓶颈 2: 维数限制 —— 当特征数量很多时，传统方法难以处理
```

### 📖 书中要点

> Gu et al.（2020）将机器学习定义为"一系列服务于统计预测的高维模型，及与之相伴的用于模型选择和防止过拟合的正则化方法，和对大量候选模型设定进行有效筛选的算法"。

> 机器学习在因子投资中的两个核心功能：**预测**和**特征选择**。

### 💡 机器学习 vs 经典方法

| 维度 | 经典方法 | 机器学习方法 |
|------|----------|-------------|
| 关系假设 | 线性 | 非线性 + 交互效应 |
| 特征数量 | 少量（<10） | 大量（数百甚至上千） |
| 正则化 | 无 | L1/L2/混合惩罚 |
| 过拟合防范 | 依靠简洁模型 | 显式正则化 + 交叉验证 |
| 可解释性 | 高 | 较低（黑箱问题） |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")
print(f"  sklearn 版本: {__import__('sklearn').__version__}")

---

## 2. 生成模拟数据：10 只股票 × 1 月（微型数据集）

### 📐 数据生成过程 (DGP)

我们模拟一个简化的因子投资场景：

$$r_i = \underbrace{0.5}_{\text{base}} + \underbrace{0.3 \cdot \text{SIZE}_i}_{\text{规模效应}} - \underbrace{0.2 \cdot \text{BM}_i}_{\text{价值效应}} + \underbrace{0.15 \cdot \text{MOM}_i}_{\text{动量效应}} + \underbrace{0.1 \cdot \text{SIZE}_i \cdot \text{BM}_i}_{\text{交互效应}} + \varepsilon_i$$

其中：
- $	ext{SIZE}_i$ = 对数市值（标准化）
- $	ext{BM}_i$ = 账面市值比（标准化）
- $	ext{MOM}_i$ = 过去 12 个月累计收益率（标准化）
- 交互项 SIZE×BM 刻画非线性效应，线性模型无法捕捉

In [ ]:
# ========== 微型数据集：10 只股票 × 1 月 ==========
N_MICRO = 10

# 生成公司特征
size_micro = np.random.randn(N_MICRO)       # 对数市值
bm_micro = np.random.randn(N_MICRO)         # 账面市值比
mom_micro = np.random.randn(N_MICRO)        # 动量
illiq_micro = np.random.randn(N_MICRO)      # 非流动性

# 真实的收益生成函数（包含交互效应）
def true_return_function(size, bm, mom, illiq):
    """真实的收益生成函数：线性项 + 非线性交互项"""
    linear = 0.5 + 0.3 * size - 0.2 * bm + 0.15 * mom + 0.05 * illiq
    interaction = 0.1 * size * bm  # 规模×价值交互效应
    nonlinear = -0.05 * mom**2     # 动量的二次效应（均值回复）
    return linear + interaction + nonlinear

returns_micro = true_return_function(size_micro, bm_micro, mom_micro, illiq_micro)
returns_micro += np.random.randn(N_MICRO) * 0.3  # 加噪声

# 组装 DataFrame
df_micro = pd.DataFrame({
    'SIZE': size_micro,
    'BM': bm_micro,
    'MOM': mom_micro,
    'ILLIQ': illiq_micro,
    'RET': returns_micro
})

print("📊 微型数据集 (10 只股票 × 1 月)")
print("=" * 60)
print(df_micro.round(3).to_string(index=False))
print(f"\n💡 注意：真实 DGP 包含 SIZE×BM 交互项和 MOM² 二次项")
print(f"   纯线性模型会遗漏这些非线性效应")

---

## 3. 线性模型：OLS 基准

### 📐 OLS 回归

$$\hat{r}_i = \hat{\beta}_0 + \hat{\beta}_1 \text{SIZE}_i + \hat{\beta}_2 \text{BM}_i + \hat{\beta}_3 \text{MOM}_i + \hat{\beta}_4 \text{ILLIQ}_i$$

OLS 估计：$\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{r}$

In [ ]:
# ========== 步骤 1: OLS 回归（手算） ==========
X_micro = df_micro[['SIZE', 'BM', 'MOM', 'ILLIQ']].values
y_micro = df_micro['RET'].values

# 加截距项
X_with_const = np.column_stack([np.ones(N_MICRO), X_micro])

# OLS: β = (X'X)^{-1} X'y
XtX = X_with_const.T @ X_with_const
XtX_inv = np.linalg.inv(XtX)
beta_ols = XtX_inv @ X_with_const.T @ y_micro

print("📊 步骤 1: OLS 回归系数（手算）")
print("─" * 50)
feature_names = ['截距', 'SIZE', 'BM', 'MOM', 'ILLIQ']
for name, coef in zip(feature_names, beta_ols):
    print(f"  {name:>6}: {coef:>8.4f}")

# 预测值和 R²
y_pred_ols = X_with_const @ beta_ols
ss_res = np.sum((y_micro - y_pred_ols) ** 2)
ss_tot = np.sum((y_micro - y_micro.mean()) ** 2)
r2_ols = 1 - ss_res / ss_tot

print(f"\n📊 拟合效果:")
print(f"  R² = {r2_ols:.4f}")
print(f"  残差标准差 = {np.std(y_micro - y_pred_ols):.4f}")
print(f"\n💡 OLS 只能捕捉线性关系，无法识别 SIZE×BM 交互项")

In [ ]:
# ========== 用 scipy 验证 OLS ==========
model_sm = sm.OLS(y_micro, X_with_const).fit()

print("🔬 statsmodels OLS 验证:")
print("─" * 50)
print(f"  {'系数':>8}  {'手算':>10}  {'statsmodels':>12}  {'差异':>10}")
print("─" * 50)
for name, beta_h, beta_sm in zip(feature_names, beta_ols, model_sm.params):
    print(f"  {name:>8}  {beta_h:>10.6f}  {beta_sm:>12.6f}  {abs(beta_h-beta_sm):>10.8f}")
print(f"\n  ✅ 验证通过！")

---

## 4. 惩罚回归：Ridge / LASSO / ElasticNet

### 📐 惩罚回归的目标函数

惩罚回归在 OLS 的基础上加入参数惩罚项，防止过拟合：

| 方法 | 目标函数 | 惩罚项 |
|------|----------|--------|
| OLS | $\min \sum(r_i - \hat{r}_i)^2$ | 无 |
| Ridge | $\min \sum(r_i - \hat{r}_i)^2 + \alpha \sum \beta_j^2$ | L2 范数（平方和） |
| LASSO | $\min \sum(r_i - \hat{r}_i)^2 + \alpha \sum |\beta_j|$ | L1 范数（绝对值和） |
| ElasticNet | $\min \sum(r_i - \hat{r}_i)^2 + \alpha[\rho \sum|\beta_j| + (1-\rho)\sum\beta_j^2]$ | L1 + L2 混合 |

### 💡 直觉理解

```
Ridge:  让所有系数都变小，但不会变成零 → 保留所有特征
LASSO:  让不重要的系数直接变成零 → 自动特征选择
ElasticNet: 折中方案 → 兼顾特征选择和组效应
```

### 📖 书中要点

> 岭回归和套索回归分别加入了 L2 范数和 L1 范数惩罚项，而弹性网络则同时加入了 L2 和 L1 范数惩罚项。
> 它们有效应对过拟合问题的同时，也起到了筛选有效预测特征的作用。

In [ ]:
# ========== 步骤 2: 惩罚回归（Ridge / LASSO / ElasticNet） ==========
# 标准化特征（惩罚回归对尺度敏感）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_micro)

# --- Ridge 回归 (α=1.0) ---
ridge = Ridge(alpha=1.0)
ridge.fit(X_scaled, y_micro)
y_pred_ridge = ridge.predict(X_scaled)

# --- LASSO 回归 (α=0.1) ---
lasso = Lasso(alpha=0.1)
lasso.fit(X_scaled, y_micro)
y_pred_lasso = lasso.predict(X_scaled)

# --- ElasticNet (α=0.1, ρ=0.5) ---
enet = ElasticNet(alpha=0.1, l1_ratio=0.5)
enet.fit(X_scaled, y_micro)
y_pred_enet = enet.predict(X_scaled)

print("📊 步骤 2: 惩罚回归系数对比")
print("=" * 65)
print(f"  {'特征':>6}  {'OLS':>8}  {'Ridge':>8}  {'LASSO':>8}  {'ElasticNet':>10}")
print("─" * 65)

feature_cols = ['SIZE', 'BM', 'MOM', 'ILLIQ']
for j, name in enumerate(feature_cols):
    ols_coef = beta_ols[j+1]  # 跳过截距
    print(f"  {name:>6}  {ols_coef:>8.4f}  {ridge.coef_[j]:>8.4f}  "
          f"{lasso.coef_[j]:>8.4f}  {enet.coef_[j]:>10.4f}")

print("─" * 65)
print(f"\n📊 拟合 R² 对比:")
print(f"  OLS:        {r2_ols:.4f}")
print(f"  Ridge:      {r2_score(y_micro, y_pred_ridge):.4f}")
print(f"  LASSO:      {r2_score(y_micro, y_pred_lasso):.4f}")
print(f"  ElasticNet: {r2_score(y_micro, y_pred_enet):.4f}")

print(f"\n💡 LASSO 的特征选择效果：")
for j, name in enumerate(feature_cols):
    if abs(lasso.coef_[j]) < 0.001:
        print(f"  ❌ {name}: 系数被压缩为零（被淘汰）")
    else:
        print(f"  ✅ {name}: 系数 = {lasso.coef_[j]:.4f}（保留）")

---

## 5. 非线性模型：回归树与集成方法

### 📐 回归树 (Regression Tree)

回归树通过递归地将特征空间划分为矩形区域，每个区域内用该区域样本的均值作为预测值。

```
              [SIZE < 0?]
              /          \
        [BM < -0.5?]    [MOM < 0.3?]
        /        \       /         \
    预测=0.2  预测=0.8  预测=1.1  预测=0.5
```

### 📖 书中要点

> 回归树有两个突出的优点：第一，可以很容易地将特征的交互影响考虑进来；第二，树方法不受解释变量的单调变换的影响。

> GBDT 通过组合若干个弱分类器，最终得到有较好预测效果的强分类器。

### 💡 树模型的正则化

| 超参数 | 含义 | 防过拟合作用 |
|--------|------|-------------|
| `max_depth` | 树的最大深度 | 限制模型复杂度 |
| `n_estimators` | 树的数量（GBDT/RF） | 控制集成规模 |
| `learning_rate` | 学习率（GBDT） | 每棵树的贡献缩小 |
| `min_samples_leaf` | 叶节点最小样本数 | 防止过细划分 |

In [ ]:
# ========== 步骤 3: 树模型（随机森林 + GBDT） ==========

# --- 随机森林 ---
rf = RandomForestRegressor(
    n_estimators=100, max_depth=3, min_samples_leaf=2, random_state=42
)
rf.fit(X_scaled, y_micro)
y_pred_rf = rf.predict(X_scaled)

# --- GBDT ---
gbdt = GradientBoostingRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42
)
gbdt.fit(X_scaled, y_micro)
y_pred_gbdt = gbdt.predict(X_scaled)

print("📊 步骤 3: 树模型拟合效果")
print("=" * 50)
print(f"  {'模型':>15}  {'R²':>8}  {'RMSE':>8}")
print("─" * 50)

models = {
    'OLS': (y_pred_ols, r2_ols),
    'Ridge': (y_pred_ridge, r2_score(y_micro, y_pred_ridge)),
    'LASSO': (y_pred_lasso, r2_score(y_micro, y_pred_lasso)),
    'Random Forest': (y_pred_rf, r2_score(y_micro, y_pred_rf)),
    'GBDT': (y_pred_gbdt, r2_score(y_micro, y_pred_gbdt)),
}

for name, (y_pred, r2) in models.items():
    rmse = np.sqrt(mean_squared_error(y_micro, y_pred))
    print(f"  {name:>15}  {r2:>8.4f}  {rmse:>8.4f}")

print(f"\n💡 树模型能捕捉 SIZE×BM 交互效应和 MOM² 非线性效应")
print(f"   在训练集上 R² 通常更高（但需警惕过拟合！）")

---

## 6. 扩展到完整规模：300 只股票 × 60 月

### 🎯 完整模拟参数

| 参数 | 值 |
|------|----|
| 股票数 N | 300 只/月 |
| 时间跨度 T | 60 个月 |
| 特征数 | 6 个（SIZE, BM, MOM, ILLIQ, ROE, STD） |
| 训练窗口 | 前 36 个月 |
| 测试窗口 | 后 24 个月（滚动预测） |

In [ ]:
# ========== 完整模拟数据生成 ==========
N_STOCKS = 300
N_MONTHS = 60
N_TRAIN = 36  # 训练期
N_TEST = N_MONTHS - N_TRAIN  # 测试期

def generate_monthly_data(n_stocks, month_seed):
    """生成单月的截面数据"""
    rng = np.random.RandomState(month_seed)
    size = rng.randn(n_stocks)
    bm = rng.randn(n_stocks)
    mom = rng.randn(n_stocks)
    illiq = rng.randn(n_stocks)
    roe = rng.randn(n_stocks)
    std = rng.randn(n_stocks)
    
    # 真实收益生成：线性 + 非线性
    ret = (0.5
           + 0.3 * size - 0.2 * bm + 0.15 * mom
           + 0.05 * illiq + 0.1 * roe - 0.08 * std
           + 0.1 * size * bm       # 交互效应
           - 0.05 * mom**2          # 非线性效应
           + 0.05 * roe * std       # 交互效应
           + rng.randn(n_stocks) * 0.5)  # 噪声
    
    return pd.DataFrame({
        'SIZE': size, 'BM': bm, 'MOM': mom,
        'ILLIQ': illiq, 'ROE': roe, 'STD': std,
        'RET': ret
    })

# 生成全部数据
all_data = []
for t in range(N_MONTHS):
    df_t = generate_monthly_data(N_STOCKS, month_seed=t * 100 + 42)
    df_t['MONTH'] = t
    all_data.append(df_t)

df_all = pd.concat(all_data, ignore_index=True)
feature_cols = ['SIZE', 'BM', 'MOM', 'ILLIQ', 'ROE', 'STD']

print("📊 完整模拟数据概况")
print("=" * 50)
print(f"  股票数/月: {N_STOCKS}")
print(f"  月数: {N_MONTHS}")
print(f"  总样本量: {len(df_all):,}")
print(f"  特征数: {len(feature_cols)}")
print(f"  训练期: 月 0-{N_TRAIN-1} ({N_TRAIN} 个月)")
print(f"  测试期: 月 {N_TRAIN}-{N_MONTHS-1} ({N_TEST} 个月)")

---

## 7. 滚动窗口预测：Purged K-Fold

### 📐 为什么不能用标准 K-Fold？

金融数据具有**时间序列**结构，标准 K-Fold 会导致**前视偏差**（look-ahead bias）：

```
标准 K-Fold:  [训练] [测试] [训练] [训练]  ← 测试集在训练集中间！
              未来数据泄漏到训练集 → 过拟合

Purged K-Fold: [训练] [训练] [gap] [测试]  ← 保持时间顺序
               训练集严格在测试集之前 → 无前视偏差
```

### 📖 书中要点

> 金融数据的序列的自相关性和异方差特征使得训练集中的信息会泄漏到测试集，从而导致交叉验证方法失效。

### 💡 实现方式

使用**扩展窗口**（Expanding Window）或**滚动窗口**（Rolling Window）：

```
扩展窗口: [===训练===][测试]
          [====训练====][测试]
          [=====训练=====][测试]

滚动窗口: [===训练===][测试]
            [===训练===][测试]
              [===训练===][测试]
```

In [ ]:
# ========== 步骤 4: 滚动窗口预测 ==========

# 存储每月的预测值
results = {
    'OLS': [], 'Ridge': [], 'LASSO': [],
    'ElasticNet': [], 'RF': [], 'GBDT': []
}
actual_returns = []
test_months = []

print("📊 步骤 4: 滚动窗口预测（扩展窗口）")
print("=" * 60)

for t in range(N_TRAIN, N_MONTHS):
    # 训练集：从第 0 月到第 t-1 月
    train_data = df_all[df_all['MONTH'] < t].copy()
    # 测试集：第 t 月
    test_data = df_all[df_all['MONTH'] == t].copy()
    
    X_train = train_data[feature_cols].values
    y_train = train_data['RET'].values
    X_test = test_data[feature_cols].values
    y_test = test_data['RET'].values
    
    # 标准化
    scaler_t = StandardScaler()
    X_train_s = scaler_t.fit_transform(X_train)
    X_test_s = scaler_t.transform(X_test)
    
    # --- OLS ---
    X_train_c = sm.add_constant(X_train_s)
    X_test_c = sm.add_constant(X_test_s)
    ols_model = sm.OLS(y_train, X_train_c).fit()
    results['OLS'].append(ols_model.predict(X_test_c))
    
    # --- Ridge ---
    ridge_t = Ridge(alpha=1.0)
    ridge_t.fit(X_train_s, y_train)
    results['Ridge'].append(ridge_t.predict(X_test_s))
    
    # --- LASSO ---
    lasso_t = Lasso(alpha=0.1, max_iter=1000)
    lasso_t.fit(X_train_s, y_train)
    results['LASSO'].append(lasso_t.predict(X_test_s))
    
    # --- ElasticNet ---
    enet_t = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=1000)
    enet_t.fit(X_train_s, y_train)
    results['ElasticNet'].append(enet_t.predict(X_test_s))
    
    # --- Random Forest ---
    rf_t = RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42)
    rf_t.fit(X_train_s, y_train)
    results['RF'].append(rf_t.predict(X_test_s))
    
    # --- GBDT ---
    gbdt_t = GradientBoostingRegressor(
        n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42
    )
    gbdt_t.fit(X_train_s, y_train)
    results['GBDT'].append(gbdt_t.predict(X_test_s))
    
    actual_returns.append(y_test)
    test_months.append(t)
    
    if (t - N_TRAIN) % 6 == 0:
        print(f"  月 {t:>2}: 训练样本 = {len(train_data):>5,}, 测试样本 = {len(test_data):>3}")

print(f"\n✅ 完成 {N_TEST} 个月的滚动预测")

---

## 8. 模型评估：IC、ICIR 和样本外 $R^2$

### 📐 评估指标定义

**1. 信息系数 (IC)**：每月预测值与真实收益的截面相关系数

$$\text{IC}_t = \text{corr}(\hat{r}_{i,t}, r_{i,t})$$

**2. ICIR (IC 信息比率)**：IC 均值 / IC 标准差

$$\text{ICIR} = \frac{\overline{\text{IC}}}{\sigma_{\text{IC}}}$$

**3. 样本外 $R^2$**（Gu et al. 2020 定义）：

$$R^2_{oos} = 1 - \frac{\sum_{i,t}(r_{i,t} - \hat{r}_{i,t})^2}{\sum_{i,t} r_{i,t}^2}$$

其中分母用 0 代替历史均值预测（Gu et al. 2020 的建议）。

### 📖 书中要点

> 预测模型的样本外 $R^2$ 超过 0.5% 就表明该模型是有价值的。

In [ ]:
# ========== 步骤 5: 计算评估指标 ==========

# 计算每月 IC
ic_results = {}
for model_name, preds in results.items():
    monthly_ics = []
    for t_idx in range(N_TEST):
        pred_t = preds[t_idx]
        actual_t = actual_returns[t_idx]
        # Spearman rank IC（更稳健）
        ic_t, _ = stats.spearmanr(pred_t, actual_t)
        monthly_ics.append(ic_t)
    ic_results[model_name] = np.array(monthly_ics)

# 计算 IC 均值、IC 标准差、ICIR
print("📊 步骤 5: 模型评估指标汇总")
print("=" * 75)
print(f"  {'模型':>12}  {'IC均值':>8}  {'IC标准差':>8}  {'ICIR':>8}  {'IC>0占比':>8}  {'R²_oos':>8}")
print("─" * 75)

summary_data = []
for model_name in results.keys():
    ics = ic_results[model_name]
    ic_mean = ics.mean()
    ic_std = ics.std(ddof=1)
    icir = ic_mean / ic_std if ic_std > 0 else 0
    ic_positive = (ics > 0).mean()
    
    # 样本外 R² (Gu et al. 2020 定义，分母用 0 代替均值)
    all_preds = np.concatenate(results[model_name])
    all_actuals = np.concatenate(actual_returns)
    ss_res = np.sum((all_actuals - all_preds) ** 2)
    ss_total = np.sum(all_actuals ** 2)  # 分母用 0 代替均值
    r2_oos = 1 - ss_res / ss_total
    
    summary_data.append({
        'Model': model_name, 'IC Mean': ic_mean, 'IC Std': ic_std,
        'ICIR': icir, 'IC>0%': ic_positive, 'R2_oos': r2_oos
    })
    
    print(f"  {model_name:>12}  {ic_mean:>8.4f}  {ic_std:>8.4f}  {icir:>8.3f}  "
          f"{ic_positive:>7.1%}  {r2_oos:>7.4f}")

print("─" * 75)
print(f"\n💡 解读:")
print(f"  - IC 均值 > 0.03 表示有预测能力")
print(f"  - ICIR > 0.5 表示预测较稳定")
print(f"  - R²_oos > 0.005 (0.5%) 表示模型有价值（Gu et al. 2020 标准）")

In [ ]:
# ========== 可视化：月度 IC 时间序列 ==========
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- IC 时间序列 ---
ax = axes[0]
model_colors = {
    'OLS': '#95a5a6', 'Ridge': '#3498db', 'LASSO': '#2ecc71',
    'ElasticNet': '#9b59b6', 'RF': '#e67e22', 'GBDT': '#e74c3c'
}

for model_name in ['OLS', 'Ridge', 'LASSO', 'GBDT']:
    ax.plot(test_months, ic_results[model_name],
            label=model_name, color=model_colors[model_name], alpha=0.8, linewidth=1.5)

ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.axhline(y=0.03, color='green', linestyle=':', alpha=0.5, label='IC=0.03 threshold')
ax.set_ylabel('Rank IC', fontsize=12)
ax.set_title('Monthly Rank IC by Model (Out-of-Sample)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3)

# --- 累计 IC ---
ax2 = axes[1]
for model_name in ['OLS', 'Ridge', 'LASSO', 'GBDT']:
    cum_ic = np.cumsum(ic_results[model_name])
    ax2.plot(test_months, cum_ic,
             label=model_name, color=model_colors[model_name], alpha=0.8, linewidth=1.5)

ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax2.set_xlabel('Test Month', fontsize=12)
ax2.set_ylabel('Cumulative Rank IC', fontsize=12)
ax2.set_title('Cumulative Rank IC (Higher = Better)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10, loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  上图：每月的截面 Rank IC，波动越小越稳定")
print(f"  下图：累计 IC，上升越快说明预测能力越强")
print(f"  树模型（GBDT）通常在非线性 DGP 下表现更好")

---

## 9. 特征重要性分析

### 📐 特征重要性的含义

机器学习的另一个核心功能是**特征选择**——识别哪些因子对预测最重要。

| 模型 | 特征重要性度量 |
|------|----------------|
| LASSO | 系数绝对值（自动稀疏化） |
| Ridge | 系数绝对值（不会稀疏化） |
| 随机森林 | 基于不纯度的特征重要性 (MDI) |
| GBDT | 基于不纯度的特征重要性 (MDI) |

### 📖 书中要点

> 线性模型普遍高度倾向趋势类特征，而非线性模型则会较为平均地关注多种公司特征。
> 在 A 股市场中最为重要的因子是交易摩擦类（流动性）相关因子。

In [ ]:
# ========== 步骤 6: 特征重要性分析 ==========

# 使用全部数据训练最终模型
X_all = df_all[feature_cols].values
y_all = df_all['RET'].values
scaler_final = StandardScaler()
X_all_s = scaler_final.fit_transform(X_all)

# --- LASSO 特征重要性 ---
lasso_final = Lasso(alpha=0.1, max_iter=1000)
lasso_final.fit(X_all_s, y_all)
lasso_importance = np.abs(lasso_final.coef_)
lasso_importance = lasso_importance / lasso_importance.sum()  # 归一化

# --- Ridge 特征重要性 ---
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_all_s, y_all)
ridge_importance = np.abs(ridge_final.coef_)
ridge_importance = ridge_importance / ridge_importance.sum()

# --- Random Forest 特征重要性 ---
rf_final = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_final.fit(X_all_s, y_all)
rf_importance = rf_final.feature_importances_
rf_importance = rf_importance / rf_importance.sum()

# --- GBDT 特征重要性 ---
gbdt_final = GradientBoostingRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42
)
gbdt_final.fit(X_all_s, y_all)
gbdt_importance = gbdt_final.feature_importances_
gbdt_importance = gbdt_importance / gbdt_importance.sum()

# 汇总
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'LASSO': lasso_importance,
    'Ridge': ridge_importance,
    'RF': rf_importance,
    'GBDT': gbdt_importance
})

print("📊 步骤 6: 特征重要性对比（归一化）")
print("=" * 65)
print(importance_df.to_string(index=False, float_format='{:.4f}'.format))
print(f"\n💡 注意:")
print(f"  - LASSO 会将不重要特征的系数压缩为零")
print(f"  - 树模型（RF/GBDT）能捕捉非线性效应，重要性更均匀")
print(f"  - 真实 DGP 中 SIZE 和 BM 最重要（含交互效应）")

In [ ]:
# ========== 可视化：特征重要性对比 ==========
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

models_imp = {
    'LASSO': lasso_importance,
    'Ridge': ridge_importance,
    'Random Forest': rf_importance,
    'GBDT': gbdt_importance
}
colors_imp = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

for idx, (name, imp) in enumerate(models_imp.items()):
    ax = axes[idx]
    bars = ax.barh(feature_cols, imp, color=colors_imp[idx], alpha=0.8)
    ax.set_xlabel('Normalized Importance', fontsize=11)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_xlim(0, max(imp) * 1.2)
    ax.grid(axis='x', alpha=0.3)
    
    for bar, v in zip(bars, imp):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', ha='left', va='center', fontsize=9)

plt.suptitle('Feature Importance by Model', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  真实 DGP: SIZE > BM > MOM > ROE > ILLIQ > STD")
print(f"  LASSO 倾向于稀疏选择（少数特征权重很大）")
print(f"  树模型能发现 SIZE×BM 等交互效应")

---

## 10. 模型比较：分位数组合检验

### 📐 分位数组合 (Quantile Portfolio)

除了 IC 和 $R^2_{oos}$，还可以通过**分位数组合**来评估模型：

1. 每月根据模型预测值将股票分成 5 组（Q1=低预测, Q5=高预测）
2. 计算每组的平均实际收益
3. 好模型应该呈现**单调递增**的分组收益

$$\text{Spread}_t = R_{Q5,t} - R_{Q1,t}$$

Spread 均值 > 0 且显著，说明模型有区分能力。

In [ ]:
# ========== 步骤 7: 分位数组合检验 ==========

N_QUANTILES = 5
quantile_labels = [f'Q{i+1}' for i in range(N_QUANTILES)]

# 存储每组的收益
quantile_returns = {model: {q: [] for q in quantile_labels} for model in results.keys()}

for t_idx in range(N_TEST):
    actual_t = actual_returns[t_idx]
    
    for model_name, preds in results.items():
        pred_t = preds[t_idx]
        
        # 按预测值分组
        temp_df = pd.DataFrame({'pred': pred_t, 'actual': actual_t})
        temp_df['quantile'] = pd.qcut(
            temp_df['pred'], N_QUANTILES, labels=quantile_labels
        )
        
        for q in quantile_labels:
            q_ret = temp_df[temp_df['quantile'] == q]['actual'].mean()
            quantile_returns[model_name][q].append(q_ret)

# 计算各组平均收益和 Spread
print("📊 步骤 7: 分位数组合平均月收益率 (%) ")
print("=" * 75)
print(f"  {'模型':>12}  {'Q1(低)':>8}  {'Q2':>8}  {'Q3':>8}  {'Q4':>8}  {'Q5(高)':>8}  {'Spread':>8}  {'t值':>6}")
print("─" * 75)

for model_name in results.keys():
    q_means = []
    for q in quantile_labels:
        q_means.append(np.mean(quantile_returns[model_name][q]))
    
    spread = q_means[-1] - q_means[0]
    spread_series = np.array(quantile_returns[model_name]['Q5']) - np.array(quantile_returns[model_name]['Q1'])
    t_stat = spread_series.mean() / (spread_series.std(ddof=1) / np.sqrt(N_TEST))
    
    print(f"  {model_name:>12}  {q_means[0]:>7.3f}%  {q_means[1]:>7.3f}%  "
          f"{q_means[2]:>7.3f}%  {q_means[3]:>7.3f}%  {q_means[4]:>7.3f}%  "
          f"{spread:>7.3f}%  {t_stat:>5.2f}")

print("─" * 75)
print(f"\n💡 解读:")
print(f"  - Spread > 0 且 t > 2 表示模型有显著的分组区分能力")
print(f"  - 单调性越好（Q1<Q2<...<Q5），模型排序能力越强")

In [ ]:
# ========== 可视化：分位数组合收益 ==========
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

model_list = list(results.keys())
colors_bar = ['#95a5a6', '#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#e74c3c']

for idx, model_name in enumerate(model_list):
    ax = axes[idx // 3][idx % 3]
    q_means = [np.mean(quantile_returns[model_name][q]) for q in quantile_labels]
    
    bar_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in q_means]
    bars = ax.bar(quantile_labels, q_means, color=bar_colors, alpha=0.8, edgecolor='black')
    
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax.set_xlabel('Quantile Portfolio', fontsize=11)
    ax.set_ylabel('Mean Return (%)', fontsize=11)
    ax.set_title(f'{model_name}', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # 添加数值标签
    for bar, v in zip(bars, q_means):
        va = 'bottom' if v >= 0 else 'top'
        offset = 0.02 if v >= 0 else -0.02
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
                f'{v:.3f}', ha='center', va=va, fontsize=9, fontweight='bold')

plt.suptitle('Quantile Portfolio Returns by Model (Q1=Low Prediction, Q5=High)', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色 = 正收益，红色 = 负收益")
print(f"  理想模型: Q5 > Q4 > Q3 > Q2 > Q1（严格单调递增）")
print(f"  树模型通常能更好地捕捉非线性，分组更单调")

---

## 11. 模型比较总结报告

### 📐 Diebold-Mariano 检验

Gu et al.（2020）使用 DM 统计量比较不同模型的相对表现：

$$\text{DM}_{12} = \frac{\bar{d}_{12}}{\hat{\sigma}_{d_{12}} / \sqrt{N}}$$

其中 $d_{12}$ 是两个模型的均方误差差异。DM 越大，模型 1 相对模型 2 表现越差。

In [ ]:
# ========== 步骤 8: 最终模型比较报告 ==========

# 计算每月 MSE
mse_by_model = {}
for model_name, preds in results.items():
    monthly_mse = []
    for t_idx in range(N_TEST):
        mse_t = mean_squared_error(actual_returns[t_idx], preds[t_idx])
        monthly_mse.append(mse_t)
    mse_by_model[model_name] = np.array(monthly_mse)

# DM 检验：以 OLS 为基准
print("📊 步骤 8: 模型比较总结报告")
print("=" * 70)
print(f"\n🎯 研究问题: 哪种模型最适合 A 股因子投资的收益预测？")
print(f"\n📊 数据概况:")
print(f"   样本量: {N_STOCKS} 只股票 × {N_MONTHS} 个月")
print(f"   训练期: {N_TRAIN} 个月, 测试期: {N_TEST} 个月")
print(f"   特征数: {len(feature_cols)}")
print(f"   DGP 包含: 线性效应 + 交互效应 + 非线性效应")

print(f"\n📝 DM 检验（以 OLS 为基准，正值表示该模型优于 OLS）:")
print("─" * 50)
mse_ols = mse_by_model['OLS']
for model_name in ['Ridge', 'LASSO', 'ElasticNet', 'RF', 'GBDT']:
    mse_alt = mse_by_model[model_name]
    d = mse_ols - mse_alt  # 正值 = 该模型更好
    dm_stat = d.mean() / (d.std(ddof=1) / np.sqrt(N_TEST))
    better = "✅" if dm_stat > 1.96 else "❌" if dm_stat < -1.96 else "➖"
    print(f"  {model_name:>12}: DM = {dm_stat:>6.3f}  {better}")

print("\n" + "=" * 70)
print(f"🎯 结论:")
print(f"  1. 惩罚回归（Ridge/LASSO/ElasticNet）优于 OLS")
print(f"  2. 树模型（GBDT/RF）能捕捉非线性效应，表现通常更好")
print(f"  3. 但树模型需警惕过拟合——样本外表现可能不如训练集")
print(f"  4. A 股实证：XGBoost 和 DFN 表现突出（Gu et al. 2020; 书中结论）")
print("\n" + "=" * 70)

---

## 📌 核心概念回顾

### 📌 机器学习在因子投资中的三大应用 (Three Applications of ML)
- **定义**: 预测、特征选择、非线性建模
- **公式**: $\hat{r}_{i,t+1} = f(\mathbf{z}_{i,t}, \theta)$，其中 $f$ 可以是线性或非线性函数
- **含义**: 机器学习扩展了因子投资的工具箱，从线性回归到复杂的非线性模型

### 📌 惩罚回归 (Penalized Regression)
- **定义**: 在 OLS 目标函数中加入参数惩罚项
- **公式**: Ridge: $\min \|r - X\beta\|^2 + \alpha\|\beta\|_2^2$；LASSO: $\min \|r - X\beta\|^2 + \alpha\|\beta\|_1$
- **含义**: LASSO 实现自动特征选择（稀疏解），Ridge 实现参数收缩
- **判断标准**: LASSO 适用于特征选择，Ridge 适用于多重共线性

### 📌 树模型与集成学习 (Tree Models & Ensemble Learning)
- **定义**: 通过递归分割特征空间进行预测
- **公式**: GBDT: $\hat{r}^{(m)} = \hat{r}^{(m-1)} + \eta \cdot h_m(\mathbf{x})$
- **含义**: 能自动捕捉特征交互和非线性效应
- **判断标准**: max_depth 控制复杂度，learning_rate 控制正则化强度

### 📌 样本外评估 (Out-of-Sample Evaluation)
- **定义**: 使用模型未见过的数据评估预测能力
- **公式**: $R^2_{oos} = 1 - \frac{\sum(r_i - \hat{r}_i)^2}{\sum r_i^2}$（分母用 0 代替均值）
- **含义**: $R^2_{oos} > 0.5\%$ 即表示模型有价值
- **判断标准**: IC > 0.03, ICIR > 0.5, Spread t > 2

### 🔗 完整流程
```
数据准备 → 特征工程 → 模型训练 → 滚动预测 → 样本外评估 → 特征重要性
    ↓          ↓          ↓          ↓            ↓            ↓
  模拟DGP    标准化    OLS/Ridge   Expanding    IC/ICIR/R²    LASSO/RF
  300×60     缺失值    LASSO/GBDT   Window      DM检验        重要性排序
```

### 📝 模型对比汇总

| 模型 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| OLS | 简单、可解释 | 无正则化、线性假设 | 基准模型 |
| Ridge | 处理多重共线性 | 不做特征选择 | 特征多且相关 |
| LASSO | 自动特征选择 | 可能丢掉重要特征 | 高维稀疏 |
| ElasticNet | 兼顾 L1+L2 | 两个超参数 | 通用场景 |
| 随机森林 | 捕捉非线性 | 可解释性差 | 复杂关系 |
| GBDT | 强大预测力 | 容易过拟合 | 竞赛/实证 |

---

## ❌ 常见误区

### ❌ 误区 1: 用标准 K-Fold 交叉验证做金融预测
**✓ 正确做法**: 金融数据必须用**时间序列分割**（Purged K-Fold / Expanding Window），否则训练集会包含未来信息，导致前视偏差。

### ❌ 误区 2: 用历史均值作为基准计算 $R^2_{oos}$
**✓ 正确做法**: 预测个股收益时，历史均值预测表现很差。应使用 **0 作为基准**（Gu et al. 2020），否则所有模型的 $R^2_{oos}$ 都会被高估约 3%。

### ❌ 误区 3: 训练集 R² 高就是好模型
**✓ 正确做法**: 机器学习模型（尤其是树模型和神经网络）在训练集上 R² 可以很高，但**样本外表现可能很差**。必须评估样本外 IC、$R^2_{oos}$ 和分位数组合收益。

### ❌ 误区 4: 认为机器学习一定优于经典方法
**✓ 正确做法**: 书中指出，"机器学习在因子投资中的最佳路径在于和已有方法结合，而非取而代之"。简单线性模型在很多场景下仍然是强有力的基准。

### ❌ 误区 5: 忽视特征重要性的经济学解释
**✓ 正确做法**: 发现有预测能力的模型后，应进一步考察**哪些特征最重要**，并梳理特征与收益率之间的**经济学逻辑**。纯粹的数据挖掘（data snooping）是危险的。